# **Setup**

In [ ]:
!pip install pytorch_lightning wandb

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
import wandb

from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.loggers import WandbLogger

from sklearn.model_selection import train_test_split 
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, mean_absolute_error

import kagglehub
from dotenv import load_dotenv

def set_seed(seed: int = 67):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 1488
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

#### **Dataset**

In [3]:
load_dotenv() 

path = kagglehub.competition_download('ventilator-pressure-prediction')

print("Path to competition files:", path)

df_train = pd.read_csv(path+"/train.csv")
df_test  = pd.read_csv(path+"/test.csv")

Path to competition files: /root/.cache/kagglehub/competitions/ventilator-pressure-prediction


#### **Target metric**

The official metric is MAE (mean absolute error). 
    $$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$
It is easy to understand, implement and from what internet says it tends to optimize the model to median solution. But it has it's weaknesses: the error increases linearly, which means that extreme outliers are weighted the same as small errors.

I propose two metrics as an alternative :
1. MSE - it is an industry standart metric that penaltizes outliers rather than small mistakes. It can be used as a loss function for optimization and I will try to train two models for both MAE and MSE and then compare their performance.
    $$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
2. Tolerance accuracy - it is here only because I thought it would be interesting to see what fraction of time our modelpredicts the preassure in some threshold near the actual value, and then I found out there is a metric for that. I will use it mostly to get some insights during validation.
    $$\text{Tolerance Accuracy} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{I}\left(|y_i - \hat{y}_i| \le \epsilon\right)$$

In [4]:
def masked_mae(y_true, y_pred, mask):
    return (abs(y_true - y_pred) * mask).sum() / mask.sum()

def masked_mse(y_true, y_pred, mask):
    return (((y_true - y_pred)**2) * mask).sum() / mask.sum()

def masked_tol(y_true, y_pred, mask, tolerance=1):
    return ((abs(y_true - y_pred) <= tolerance) * mask).sum() / mask.sum()

#### **Splitting and Feature engineering**

In [6]:
STEPS = 80
N_LAGS = 5
N_LEADS = 4
N_DIFFS = 4
R_VALUES = [5, 20, 50]
C_VALUES = [10, 20, 50]
USE_NEW_FEATURES = True

BASE_FEATURES = ["R", "C", "time_step", "u_in", "u_out"]
LAG_FEATURES = [f"u_in_lag{n}" for n in range(1, N_LAGS+1)] 
OLD_FEATURES = BASE_FEATURES + LAG_FEATURES + ["u_in_cumsum"]

LEAD_FEATURES = [f"u_in_lead{n}" for n in range(1, N_LEADS+1)]
DIFF_FEATURES = [f"u_in_diff{n}" for n in range(1, N_DIFFS+1)]
ONEHOT_FEATURES = ([f"R_{r}" for r in R_VALUES] + [f"C_{c}" for c in C_VALUES]
                   + [f"RC_{r}_{c}" for r in R_VALUES for c in C_VALUES])
NEW_FEATURES = ["dt", "u_in_area", "u_in_max"] + LEAD_FEATURES + DIFF_FEATURES + ONEHOT_FEATURES

FEATURES = OLD_FEATURES + NEW_FEATURES if USE_NEW_FEATURES else OLD_FEATURES
TARGET = "pressure"

for d in (df_train, df_test):
    d.sort_values(["breath_id", "time_step"], inplace=True)
    d.reset_index(drop=True, inplace=True)
assert (df_train.groupby("breath_id").size() == STEPS).all()
assert (df_test.groupby("breath_id").size() == STEPS).all()

In [7]:
breath_groups = df_train[["breath_id", "R", "C"]].drop_duplicates()
breath_groups["R_C"] = breath_groups["R"].astype(str)+'_'+breath_groups["C"].astype(str)

breath_train, breath_val = train_test_split(breath_groups["breath_id"], 
                                            train_size=0.67, 
                                            stratify=breath_groups["R_C"],
                                            random_state=SEED
                                            )
df_val   = df_train[df_train["breath_id"].isin(breath_val)].copy()
df_train = df_train[df_train["breath_id"].isin(breath_train)].copy()

mask_val = (df_val["u_out"] == 0).to_numpy()

In [8]:
def add_features(df):
    by_breath = df.groupby("breath_id")
    grouped_df = by_breath["u_in"]

    for n in range(1, N_LAGS+1):
        df[f"u_in_lag{n}"] = grouped_df.shift(n).fillna(0)
    df["u_in_cumsum"] = grouped_df.cumsum()

    df["dt"] = by_breath["time_step"].diff().fillna(0)
    df["u_in_area"] = (df["u_in"] * df["dt"]).groupby(df["breath_id"]).cumsum()

    df["u_in_max"] = grouped_df.transform("max")

    for n in range(1, N_LEADS+1):
        df[f"u_in_lead{n}"] = grouped_df.shift(-n).fillna(0)
    for n in range(1, N_DIFFS+1):
        df[f"u_in_diff{n}"] = df["u_in"] - df[f"u_in_lag{n}"]

    for r in R_VALUES:
        df[f"R_{r}"] = (df["R"] == r).astype("float32")
    for c in C_VALUES:
        df[f"C_{c}"] = (df["C"] == c).astype("float32")
    for r in R_VALUES:
        for c in C_VALUES:
            df[f"RC_{r}_{c}"] = ((df["R"] == r) & (df["C"] == c)).astype("float32")
    return df

for df in [df_train, df_val, df_test]:
    df = add_features(df)

# **ML model**

In [9]:
# DATAPREP

X_train, y_train = df_train[FEATURES].to_numpy(), df_train[TARGET].to_numpy()
X_val, y_val     = df_val[FEATURES].to_numpy(), df_val[TARGET].to_numpy()
X_test = df_test[FEATURES].to_numpy()

In [10]:
# MODELING

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

wandb.finish()
wandb.login()
run = wandb.init(project="google_brain", name="ml_new_feature_eng",
                 config=dict(max_iter=100, 
                             learning_rate=0.1, 
                             max_depth=4))

ml_model = HistGradientBoostingRegressor(random_state=SEED, verbose=1, **dict(run.config))   # verbose=1: live progress line
ml_model.fit(X_train, y_train)
print("fitted,", ml_model.n_iter_, "iterations")

rows = np.random.default_rng(SEED).choice(len(X_val), 20_000, replace=False)
imp = permutation_importance(ml_model, X_val[rows], y_val[rows], scoring="neg_mean_absolute_error",
                             n_repeats=1, random_state=SEED)
imp_mean = np.clip(imp.importances_mean, 0, None)
ml_model.feature_importances_ = imp_mean / max(imp_mean.sum(), 1e-12)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bhnyshchenko (bhnyshchenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Binning 1.077 GB of training data: 8.705 s
Binning 0.120 GB of validation data: 0.872 s
Fitting gradient boosted rounds:
Fit 100 trees in 68.575 s, (1599 total leaves)
Time spent computing histograms: 44.254s
Time spent finding best splits:  0.088s
Time spent applying splits:      6.526s
Time spent predicting:           1.230s
fitted, 100 iterations


#### **ML Results**

In [11]:
ml_val_pred = ml_model.predict(X_val)

for i, pred in enumerate(ml_model.staged_predict(X_val)):
    wandb.log({"train_loss": ml_model.train_score_[i],
               "val_mae_masked": masked_mae(y_val, pred, mask_val),
               "val_mae_all": mean_absolute_error(y_val, pred)}, step=i)

table = wandb.Table(data=list(zip(FEATURES, ml_model.feature_importances_)),
                    columns=["feature", "importance"])
wandb.log({"feature_importance": wandb.plot.bar(table, "feature", "importance")})

run.summary["val_mae"] = masked_mae(y_val, ml_val_pred, mask_val)
run.summary["val_mse"] = masked_mse(y_val, ml_val_pred, mask_val) 
run.summary["val_tol"] = masked_tol(y_val, ml_val_pred, mask_val, 1)

print(f"MeanAE: {run.summary["val_mae"]}")
print(f"MeanSE: {run.summary["val_mse"]}")
print(f"TolAcc: {run.summary["val_tol"]}")

wandb.finish()

MeanAE: 1.7968767494024067
MeanSE: 6.044904218200975
TolAcc: 0.4087063620260164


train_loss,▁▅▆▆▆▇▇▇▇▇▇█████████████████████████████
val_mae_all,█▇▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae_masked,█▇▇▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,-1.45348
val_mae,1.79688
val_mae_all,1.03291
val_mae_masked,1.79688
val_mse,6.0449
val_tol,0.40871


In [12]:
from tqdm.auto import tqdm

def predict_in_chunks(model, X, chunk=500_000):
    out = np.empty(len(X), dtype=np.float32)
    for start in tqdm(range(0, len(X), chunk), desc="ML predict", unit="chunk"):
        out[start:start + chunk] = model.predict(X[start:start + chunk])
    return out

ml_test_pred = predict_in_chunks(ml_model, X_test)

submission = pd.DataFrame({
    'id': df_test['id'],
    'pressure': ml_test_pred
})
submission.to_csv('ml_submission.csv', index=False)

print(f"Shape: {submission.shape}")
print(submission.head())


ML predict:   0%|          | 0/9 [00:00<?, ?chunk/s]

Shape: (4024000, 2)
   id  pressure
0   1  5.788988
1   2  5.401394
2   3  6.264333
3   4  7.783061
4   5  9.950681


In [19]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f ml_submission.csv -m "ml_new_features"

100% 66.3M/66.3M [00:01<00:00, 47.1MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction

# **DL model**

In [10]:
# DATA PREP

class BreathDataset(Dataset):
    def __init__(self, df, features=FEATURES, target=None, steps=STEPS):
        self.X = torch.tensor(df[features].to_numpy(), dtype=torch.float32).reshape(-1, steps, len(features))
        self.y = None if target is None else torch.tensor(df[target].to_numpy(), dtype=torch.float32).reshape(-1, steps)
        self.mask = torch.tensor((df["u_out"] == 0).to_numpy()).reshape(-1, steps)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx], self.mask[idx]
        else:
            return self.X[idx], self.y[idx], self.mask[idx]
        
train_dataset = BreathDataset(df_train, target=TARGET)
val_dataset   = BreathDataset(df_val,   target=TARGET)
test_dataset  = BreathDataset(df_test)

train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=128)

In [11]:
# CUSTOM LAYER AND OPTIM

class MyLittleLayerFriendshipIsMagic(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        he_init_std = (2/(in_features))**0.5
        self.weight = nn.Parameter(nn.init.normal_(torch.empty(out_features, in_features),
                                                    mean = 0,
                                                    std = he_init_std))
        self.bias   = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return torch.matmul(x, self.weight.T) + self.bias


class TheAmazingWorldOfOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None             
        if closure is not None:
            with torch.enable_grad(): #ya hz what this is for, something to do with lightning
                loss = closure()


        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["beta1"], group["beta2"]
            weight_decay = group["weight_decay"]
            eps = group["eps"]

            for param in group["params"]:
                if param.grad is None:
                    continue
                grad = param.grad

                param.mul_(1 - lr * weight_decay) # AdamW stuff

                state = self.state[param]
                if len(state) == 0:
                    state["step"]       = 0
                    state["exp_avg"]    = torch.zeros_like(param) # m from Adam formula
                    state["exp_avg_sq"] = torch.zeros_like(param) # v
                exp_avg    = state["exp_avg"] 
                exp_avg_sq = state["exp_avg_sq"]
                state["step"] += 1
                t = state["step"]

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)  # розібратись with this shi
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1 ** t
                bias_correction2 = 1 - beta2 ** t

                denom = exp_avg_sq.sqrt() / bias_correction2 ** 0.5 + eps
                param.addcdiv_(exp_avg, denom, value=-lr / bias_correction1)
        
        return loss

In [12]:
# MODELING

class PressureMLP(nn.Module):
    def __init__(self, in_features):
        super(PressureMLP, self).__init__()
        self.input_norm = nn.BatchNorm1d(in_features)
        self.layers = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(),
            nn.Linear(16, 32), # Here was my little layer
            nn.ReLU(),
            nn.Linear(32, 64), # Here was my little layer
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        x = self.input_norm(x.transpose(1, 2)).transpose(1, 2)
        return self.layers(x)

In [13]:
class LitPressureMLP(pl.LightningModule):
    def __init__(self, in_features, lr=1e-3, loss_fn=nn.L1Loss(reduction="none")):
        super().__init__()
        self.model = PressureMLP(in_features)
        self.loss_fn = loss_fn
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y, mask = batch
        preds = self(x).squeeze(-1)
        loss = (self.loss_fn(preds, y) * mask).sum() / mask.sum()
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, mask = batch
        preds = self(x).squeeze(-1)
        
        self.log_dict({"val_mae":masked_mae(y, preds, mask), 
                       "val_mse":masked_mse(y, preds, mask), 
                       "val_tol":masked_tol(y, preds, mask, 1)}, 
                        prog_bar=True, batch_size=int(mask.sum()))

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.estimated_stepping_batches,   
            eta_min=1e-5,                                    
        )
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}

dl_model = LitPressureMLP(in_features=len(FEATURES))

In [ ]:
wandb.login()
wandb_logger = WandbLogger(project="google_brain", name="mae_new_feature_eng")
wandb_logger.watch(dl_model, log='gradients', log_freq=50)

trainer = pl.Trainer(
    max_epochs=16,
    val_check_interval=0.5,
    log_every_n_steps=10,
    accelerator="auto",
    logger=wandb_logger,
)
trainer.fit(model=dl_model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bhnyshchenko (bhnyshchenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_ligh

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ PressureMLP │ 15.8 K │ train │     0 │
│ 1 │ loss_fn │ L1Loss      │      0 │ train │     0 │
└───┴─────────┴─────────────┴────────┴───────┴───────┘

Trainable params: 15.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.8 K                                                                                               
Total estimated model params size (MB): 0.063                                                                      
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=16` reached.


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇████
train_loss,█▆▃▃▃▂▅▃▃▅▃▃▂▃▂▃▂▂▃▂▂▄▃▂▂▂▃▂▂▂▂▂▁▂▃▂▁▁▂▂
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▄▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇███
val_mae,█▃▃▂▂▂▂▂▂▂▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mse,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_tol,▁▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▆█▇██▇██████████
epoch,15
train_loss,1.52982
trainer/global_step,6319
val_mae,1.45651
val_mse,4.59957


#### **DL Resulsts**

In [ ]:
import gc, glob, os
gc.collect()

ckpts = sorted(glob.glob("google_brain/*/checkpoints/*.ckpt"), key=os.path.getmtime)
dl_model = LitPressureMLP.load_from_checkpoint(ckpts[-1], in_features=len(FEATURES))  
dl_model.to(device).eval()
print("using", ckpts[-1], "on", dl_model.device)

y_val = df_val[TARGET].to_numpy()   

def predict_in_batches(model, dataset, batch_size=512):
    out = []
    with torch.no_grad():
        for start in range(0, len(dataset), batch_size):
            x = dataset.X[start:start + batch_size].to(model.device)
            out.append(model(x).squeeze(-1).cpu())
    return torch.cat(out).reshape(-1).numpy()

dl_val_pred  = predict_in_batches(dl_model, val_dataset)
dl_test_pred = predict_in_batches(dl_model, test_dataset)

print(f"MeanAE: {masked_mae(y_val, dl_val_pred, mask_val)}")
print(f"MeanSE: {masked_mse(y_val, dl_val_pred, mask_val)}")
print(f"TolAcc: {masked_tol(y_val, dl_val_pred, mask_val, 1)}")

pd.DataFrame({'id': df_test['id'], 'pressure': dl_test_pred}).to_csv('dl_submission.csv', index=False)
save_preds("dl", dl_val_pred, dl_test_pred)     # needs the helper cell from earlier; saves to Drive


using google_brain/i4dkvsnj/checkpoints/epoch=15-step=6319.ckpt on cuda:0
MeanAE: 1.4565101337221964
MeanSE: 4.599566856361158
TolAcc: 0.5124863243758823
saved dl: val 1991920, test 4024000 rows -> /content/drive/MyDrive/google_brain_preds


In [ ]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f dl_submission.csv -m "dl_new_feature_eng"

100% 66.8M/66.8M [00:01<00:00, 48.6MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction

# **LightGBM**

In [ ]:
!pip install -q lightgbm

In [10]:
import lightgbm as lgb
from tqdm.auto import tqdm

LGB_PARAMS = dict(objective="huber", alpha=1.0, metric="mae",
                  learning_rate=0.2, num_leaves=96, min_data_in_leaf=64,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l2=1.0, verbosity=-1, seed=SEED)
LGB_MAX_ROUNDS = 3000  
LGB_PATIENCE = 100 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bhnyshchenko (bhnyshchenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LightGBM:   0%|          | 0/3000 [00:00<?, ?tree/s]

stopped at 3000 trees, best val MAE on scored rows: 0.5948


predicting:   0%|          | 0/4 [00:00<?, ?chunk/s]

predicting:   0%|          | 0/9 [00:00<?, ?chunk/s]

MeanAE: 0.5947695238091354
MeanSE: 0.902960149325455
TolAcc: 0.8275449228542665


val_mae_scored,█▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,0.59477
val_mae_scored,0.59477
val_mse,0.90296
val_tol,0.82754


In [ ]:
def progress_callback(total, run):
    bar = tqdm(total=total, desc="LightGBM", unit="tree")
    def _callback(env):
        val_mae = env.evaluation_result_list[0][2]
        bar.update(1)
        bar.set_postfix(val_mae=f"{val_mae:.4f}")
        run.log({"val_mae_scored": val_mae}, step=env.iteration)
    _callback.order = 20      # runs before early stopping
    _callback.bar = bar
    return _callback


def predict_in_chunks(model, df, chunk=500_000):
    out = np.empty(len(df), dtype=np.float32)
    for start in tqdm(range(0, len(df), chunk), desc="predicting", unit="chunk"):
        part = df.iloc[start:start + chunk][FEATURES].to_numpy(np.float32)
        out[start:start + chunk] = model.predict(part, num_iteration=model.best_iteration)
    return out


mask_train = (df_train["u_out"] == 0).to_numpy()      
d_train = lgb.Dataset(df_train.loc[mask_train, FEATURES].to_numpy(np.float32),
                      df_train.loc[mask_train, TARGET].to_numpy())
d_val   = lgb.Dataset(df_val.loc[mask_val, FEATURES].to_numpy(np.float32),
                      df_val.loc[mask_val, TARGET].to_numpy(), reference=d_train)

wandb.finish()
lgb_run = wandb.init(project="google_brain", name="lgbm_new_feature_eng",
                     config={**LGB_PARAMS, "max_rounds": LGB_MAX_ROUNDS, "n_features": len(FEATURES)})

progress = progress_callback(LGB_MAX_ROUNDS, lgb_run)
lgb_model = lgb.train(LGB_PARAMS, d_train, num_boost_round=LGB_MAX_ROUNDS, valid_sets=[d_val],
                      callbacks=[lgb.early_stopping(LGB_PATIENCE, verbose=False), progress])
progress.bar.close()
print(f"stopped at {lgb_model.best_iteration} trees, best val MAE on scored rows: "
      f"{lgb_model.best_score['valid_0']['l1']:.4f}")
del d_train, d_val

lgb_val_pred  = predict_in_chunks(lgb_model, df_val)
lgb_test_pred = predict_in_chunks(lgb_model, df_test)

lgb_run.summary["val_mae"] = masked_mae(y_val, lgb_val_pred, mask_val)
lgb_run.summary["val_mse"] = masked_mse(y_val, lgb_val_pred, mask_val)
lgb_run.summary["val_tol"] = masked_tol(y_val, lgb_val_pred, mask_val, 1)
print(f"MeanAE: {lgb_run.summary['val_mae']}")
print(f"MeanSE: {lgb_run.summary['val_mse']}")
print(f"TolAcc: {lgb_run.summary['val_tol']}")

table = wandb.Table(data=list(zip(FEATURES, lgb_model.feature_importance("gain"))),
                    columns=["feature", "importance"])
wandb.log({"feature_importance": wandb.plot.bar(table, "feature", "importance")})
wandb.finish()

In [11]:
lgb_submission = pd.DataFrame({"id": df_test["id"], "pressure": lgb_test_pred})
assert len(lgb_submission) == len(df_test) and lgb_submission["pressure"].notna().all()
lgb_submission.to_csv("lgb_submission.csv", index=False)
print(lgb_submission.shape)             # expect (4024000, 2)
print(lgb_submission.head())

(4024000, 2)
   id  pressure
0   1  6.220419
1   2  5.763947
2   3  6.595357
3   4  7.548562
4   5  8.878742


In [ ]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f lgb_submission.csv -m "lightgbm huber, new features"

100% 66.9M/66.9M [00:05<00:00, 12.6MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction

# **Ensamble**

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor

def to_ensamble_data(df, ml_pred, dl_pred, lgb_pred):
    return pd.DataFrame({"ml": ml_pred, "dl": dl_pred, "lgb": lgb_pred,
                         "R": df["R"].to_numpy(), "C": df["C"].to_numpy(),
                         "u_out": df["u_out"].to_numpy(), "time_step": df["time_step"].to_numpy()})

meta_val  = to_ensamble_data(df_val,  ml_val_pred,  dl_val_pred,  lgb_val_pred)
meta_test = to_ensamble_data(df_test, ml_test_pred, dl_test_pred, lgb_test_pred)
groups    = df_val["breath_id"].to_numpy()

In [ ]:
mean_pred = (ml_val_pred + dl_val_pred + lgb_val_pred) / 3

grid = np.round(np.arange(0, 1.0001, 0.05), 2)
y_s, ml_s, dl_s, lgb_s = y_val[mask_val], ml_val_pred[mask_val], dl_val_pred[mask_val], lgb_val_pred[mask_val]
best_mae, best_w = np.inf, None
for w_ml in grid:
    for w_dl in grid[grid <= 1 - w_ml + 1e-9]:
        w_lgb = round(1 - w_ml - w_dl, 2)
        mae = np.abs(y_s - (w_ml * ml_s + w_dl * dl_s + w_lgb * lgb_s)).mean()
        if mae < best_mae:
            best_mae, best_w = mae, (w_ml, w_dl, w_lgb)
weighted_pred = best_w[0] * ml_val_pred + best_w[1] * dl_val_pred + best_w[2] * lgb_val_pred
print(f"best weights (ml, dl, lgb): {best_w}, masked MAE {best_mae:.4f}")

best weights (ml, dl, lgb): (np.float64(0.0), np.float64(0.0), np.float64(1.0)), masked MAE 0.5948


In [ ]:
oof = np.zeros(len(meta_val))
for fold, (tr_idx, te_idx) in enumerate(GroupKFold(n_splits=5).split(meta_val, y_val, groups=groups)):
    fold_model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=100, random_state=SEED)
    stacker = fold_model.fit(meta_val.iloc[tr_idx], y_val[tr_idx])
    oof[te_idx] = stacker.predict(meta_val.iloc[te_idx])
    print(f"fold {fold}: masked MAE = {masked_mae(y_val[te_idx], oof[te_idx], mask_val[te_idx]):.4f}")

all_preds = {"ML": ml_val_pred, "DL": dl_val_pred, "LightGBM": lgb_val_pred,
             "mean (3 models)": mean_pred, "weighted (ml/dl/lgb)": weighted_pred,
             "stacking (OOF)": oof}
results = pd.DataFrame({
    "model":      list(all_preds),
    "masked_mae": [masked_mae(y_val, p, mask_val) for p in all_preds.values()],
    "masked_mse": [masked_mse(y_val, p, mask_val) for p in all_preds.values()],
    "tol@1":      [masked_tol(y_val, p, mask_val, 1) for p in all_preds.values()],
}).sort_values("masked_mae").reset_index(drop=True)
print(results.round(4).to_string(index=False))

mae_by_model = results.set_index("model")["masked_mae"]
singles = mae_by_model[["ML", "DL", "LightGBM"]]
best_single = singles.idxmin()
gain = (1 - mae_by_model["stacking (OOF)"] / singles.min()) * 100
print(f"\nSingle models (masked MAE): " + ", ".join(f"{m} {v:.4f}" for m, v in singles.items()))
print(f"Best single model: {best_single} ({singles.min():.4f}); weights of the weighted average (ml, dl, lgb): {best_w}")
print(f"Stacking (OOF): {mae_by_model['stacking (OOF)']:.4f}, improvement over the best single model: {gain:+.1f}%")

fold 0: masked MAE = 0.6582
fold 1: masked MAE = 0.6613
fold 2: masked MAE = 0.6535
fold 3: masked MAE = 0.6613
fold 4: masked MAE = 0.6461
               model  masked_mae  masked_mse  tol@1
            LightGBM      0.5948      0.9030 0.8275
weighted (ml/dl/lgb)      0.5948      0.9030 0.8275
      stacking (OOF)      0.6561      1.2059 0.8072
     mean (3 models)      1.1091      2.5409 0.5991
                  DL      1.4565      4.5996 0.5125
                  ML      1.7969      6.0449 0.4087

Single models (masked MAE): ML 1.7969, DL 1.4565, LightGBM 0.5948
Best single model: LightGBM (0.5948); weights of the weighted average (ml, dl, lgb): (np.float64(0.0), np.float64(0.0), np.float64(1.0))
Stacking (OOF): 0.6561, improvement over the best single model: -10.3%


In [ ]:
ens_model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=100, random_state=SEED)
ens_model.fit(meta_val, y_val)         # all validation breaths
ensamble_test_pred = ens_model.predict(meta_test)

pd.DataFrame({"id": df_test["id"], "pressure": ensamble_test_pred}).to_csv("ensamble_submission.csv", index=False)
print(pd.read_csv("ensamble_submission.csv").shape)             # expect (4024000, 2)

(4024000, 2)


In [ ]:
# !kaggle competitions submit -c ventilator-pressure-prediction -f ensamble_submission.csv -m "stacking HistGradBoost ml+dl+lgb"

100% 99.5M/99.5M [00:02<00:00, 50.2MB/s]
Successfully submitted to Google Brain - Ventilator Pressure Prediction